# Day 57: Build the Core First

**Today's question:** how do you build an AI app so that most of it works, and is tested, **before** any AI
is involved?

Worked example: the *Spending Summary Assistant* we scoped on Day 56. Today we build its core.
Before running a cell marked 🤔, predict what it will show.

## 1. Why the core comes first

```
main.py       the CLI: input() and print() only
   │
spending.py   the core: load, clean, calculate        ← most bugs live here: test it hard
   │
summary.py    ONE function that will use AI           ← today: a simple template ("stub")
```

The AI part is the slowest, least predictable and only paid part of the app. The core is **deterministic**:
the same input always gives the same output, so tests can check exact answers. Building it first means you
always have a working app to demo, even if the AI part goes wrong.

## 2. Loading data into simple records

Small, strict helper first: one job, a clear error for bad input.

In [ ]:
def parse_amount(text):
    """Turn text such as '12.50' into 12.5. Raise ValueError if it is not a positive number."""
    amount = float(text)  # float("abc") raises ValueError by itself
    if amount <= 0:
        raise ValueError(f"Amount must be positive, got {amount}")
    return amount


print(parse_amount("12.50"))
print(parse_amount(" 7 "))

🤔 **Predict:** what happens with `parse_amount("-4")`?

In [ ]:
parse_amount("-4")

That error is **on purpose**: the core refuses bad data loudly. The loader decides what to do with it
(here: skip the row and count it).

In [ ]:
import csv
from pathlib import Path

DATA_PATH = Path("../../../datasets/expenses.csv")


def load_expenses(path=DATA_PATH):
    """Read the expenses CSV. Returns (expenses, skipped): a list of dicts and the number of bad rows."""
    expenses = []
    skipped = 0
    with open(path, encoding="utf-8", newline="") as file:
        for row in csv.DictReader(file):
            try:
                amount = parse_amount(row["amount"])
            except ValueError:
                skipped += 1
                continue
            expenses.append({"date": row["date"], "category": row["category"].strip().lower(), "amount": amount})
    return expenses, skipped


expenses, skipped = load_expenses()
print(len(expenses), "expenses,", skipped, "skipped")
print(expenses[:2])

Notice what is **not** loaded: the descriptions. The app never needs them, so they never enter the program
(Day 56's privacy check, turned into code).

Why plain dicts and `csv` instead of pandas? Both are fine for a capstone. Functions that take a **list of dicts**
are very easy to test with three hand-made records, as you will see in section 4.

## 3. Core logic as pure functions

A **pure function** only uses its parameters and returns a result: no `input()`, no `print()`, no files.
Same input → same output, which makes it trivial to test.

In [ ]:
def filter_month(expenses, month):
    """Keep only the expenses of one month, given as 'YYYY-MM'."""
    return [expense for expense in expenses if expense["date"].startswith(month)]


def category_totals(expenses):
    """Total amount per category, rounded to cents."""
    totals = {}
    for expense in expenses:
        category = expense["category"]
        totals[category] = totals.get(category, 0) + expense["amount"]
    return {category: round(total, 2) for category, total in totals.items()}

🤔 **Predict** the result for this tiny, hand-made sample before running:

In [ ]:
sample = [
    {"date": "2026-09-01", "category": "rent", "amount": 650.0},
    {"date": "2026-09-02", "category": "food", "amount": 12.5},
    {"date": "2026-09-03", "category": "food", "amount": 7.5},
    {"date": "2026-10-01", "category": "transport", "amount": 3.0},
]
category_totals(filter_month(sample, "2026-09"))

In [ ]:
FIXED_COSTS = {"rent", "utilities"}


def get_amount(pair):
    """Sort key: the amount in a (category, amount) pair."""
    return pair[1]


def top_categories(totals, n=3, skip_fixed=False):
    """The n biggest categories, biggest first."""
    pairs = []
    for category, total in totals.items():
        if skip_fixed and category in FIXED_COSTS:
            continue
        pairs.append((category, total))
    return sorted(pairs, key=get_amount, reverse=True)[:n]


september_totals = category_totals(filter_month(expenses, "2026-09"))
print(top_categories(september_totals))
print(top_categories(september_totals, n=2, skip_fixed=True))

## 4. Checking as you go

With tiny hand-made data we can work out every answer in our heads, so `assert` (Day 29) can check exact
values. These lines become pytest tests in section 7.

In [ ]:
assert parse_amount("12.50") == 12.5
assert category_totals(sample) == {"rent": 650.0, "food": 20.0, "transport": 3.0}
assert filter_month(sample, "2025-01") == []
assert top_categories(category_totals(sample), n=1, skip_fixed=True) == [("food", 20.0)]
print("all checks passed")

🤔 **Predict:** edge cases. What *should* these return? Decide first, then run.

In [ ]:
print(category_totals([]))
print(top_categories({}, n=3))

Empty in, empty out, no crash. Deciding edge-case behaviour **before** coding is part of the core's design.

## 5. The AI boundary: a stub that already works

The AI will write the summary. Today we write the **same function** with a plain template inside. It already
returns the exact shape the AI version will return: `{"summary": text, "tips": [tip, tip]}`.

In [ ]:
TIPS = {
    "food": "Plan a few meals ahead and cook in batches; eating out adds up fast.",
    "transport": "Compare a monthly pass with what you paid for single tickets and taxis.",
    "entertainment": "Pick one or two paid activities per week and look for free events.",
    "shopping": "Wait 48 hours before buying anything that is not essential.",
}
GENERAL_TIP = "Check your spending once a week, not only at the end of the month."


def write_summary(totals):
    """Return {"summary": text, "tips": [tip, tip]}. Day 57: template. Day 58: AI inside."""
    total = sum(totals.values())
    biggest, biggest_amount = top_categories(totals, n=1)[0]
    summary = f"You spent {total:.2f} in total. The biggest category was {biggest} ({biggest_amount:.2f})."
    tips = []
    for category, amount in top_categories(totals, n=2, skip_fixed=True):
        tips.append(TIPS.get(category, GENERAL_TIP))
    while len(tips) < 2:
        tips.append(GENERAL_TIP)
    return {"summary": summary, "tips": tips}


write_summary(september_totals)

🤔 **Predict:** tomorrow the inside of `write_summary` calls a model. Which other functions have to change?
(None. That is the point of **one function boundary**.)

## 6. Putting it together

`main()` only asks, calls and prints. Here is its body for one month:

In [ ]:
month = "2026-11"  # pretend the user typed this

totals = category_totals(filter_month(expenses, month))
for category, total in top_categories(totals, n=len(totals)):
    print(f"{category:<15}{total:>10.2f}")
result = write_summary(totals)
print("\nSummary:", result["summary"])
for tip in result["tips"]:
    print(" Tip:", tip)

## 7. From notebook to files

The same code lives in `example_app/` next to this notebook:

```
example_app/
├── spending.py        core: parse_amount, load_expenses, filter_month, category_totals, top_categories
├── summary.py         write_summary (template stub)
├── main.py            CLI loop: asks for a month, prints table + summary
└── test_spending.py   8 pytest tests
```

In the terminal:

```
cd weeks/week-12/day-57-capstone-core/example_app
uv run pytest -q          # 8 passed
uv run python main.py     # type 2026-09, then quit
```

A test is the section 4 `assert`, moved into a function whose name starts with `test_`:

```python
def test_category_totals_adds_up_each_category():
    assert category_totals(SAMPLE) == {"rent": 650.0, "food": 20.0, "transport": 3.0}
```

## 8. Common mistakes (these cells fail on purpose)

**Mistake 1:** a core function that prints instead of returning. It looks fine on screen, but returns `None`.

In [ ]:
def total_spent(expenses):
    print(sum([expense["amount"] for expense in expenses]))

In [ ]:
total_spent(sample) + 10

`TypeError: unsupported operand type(s) for +: 'NoneType' and 'int'`. Core functions **return**; only `main()`
prints.

**Mistake 2:** comparing money that was never rounded.

In [ ]:
print(0.1 + 0.2)

In [ ]:
assert 0.1 + 0.2 == 0.3

Floats are stored in binary, so `0.1 + 0.2` is `0.30000000000000004`. That is why `category_totals` rounds to
cents, and why tests compare rounded values.

| Mistake | Symptom | Fix |
|---------|---------|-----|
| `print` in a core function | `None` returned, `TypeError` later | `return` the value |
| floats compared without rounding | test fails by 0.00000000000004 | `round(x, 2)` in the core |
| tests only use the big real file | you cannot tell what the right answer is | tiny hand-made data + one real-file test |
| model call inside the loader | nothing runs offline, tests are slow | AI only inside the one boundary function |

## Recap

1. Build and test the **non-AI core first**; it is where most bugs live.
2. Core functions are **pure**: parameters in, value out, no `input`/`print`.
3. Tiny hand-made data makes exact `assert`s easy; then move them into `test_*.py` for pytest.
4. The AI part is **one function** that already works today as a template stub with the final return shape.

**Next time:** put a real (or mock) model inside that one function, and test it without the network.